# First real-data CNN smoke experiment — four classes
Select **Runtime → Change runtime type → T4 GPU**, then run all cells in order with the defaults. Authorize Drive mounting. Notebook 01 must already have saved canonical schema-v2 inspection artifacts. This notebook reuses them: it does not repeat full video decoding and duplicate hashing.

Automatic preparation writes a candidate split, readable validation report, source-review CSV, and visual review sheet. **Automatic PASS is not human approval.** Review source conventions, category mapping and related incidents; the code cannot verify semantic labels or incident independence. Both human-review and training flags default to False. No training occurs on a default Run all. After reviewing the report and sheet, change the two flags in the launch cell and run that cell, followed by the result cell.

The launcher is bounded to **2 epochs, 3 original sources/class, one clip/source** (12 clips total). No full training is launched. The CNN has fresh random weights; no pretrained model or inference API is used. One held-out source per class gives a wiring check, not reliable model accuracy. GPU availability is checked again before training; this notebook never silently substitutes CPU. Actual checkpoints/metrics are created only by executing the reviewed training call. Each run has its own Drive folder.

Saved absolute paths from another KaggleHub download are resolved safely before validation. Recognized wrapper folders are normalized only within this dataset's package. A moved download is rebound into smoke_preparation/resolved_inspection only after canonical identities, existence, copy counts and saved SHA-256 evidence match. Original Drive inspection files stay unchanged. Missing evidence/content changes require rerunning notebook 01; containment and review gates remain enforced.


In [ ]:
from pathlib import Path
import os, subprocess, sys
REPO = Path('/content/AI-Crime-Investigation-FYP')
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/ranaalihaider99993-source/AI-Crime-Investigation-FYP.git', str(REPO)], check=True)
else:
    if subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError('Colab checkout has local changes. Preserve them before updating; use a fresh runtime if needed.')
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'merge', '--ff-only', 'origin/main'], check=True)
os.chdir(REPO)
# Evict previously imported project modules after updating a reused Colab runtime.
for module_name in list(sys.modules):
    if module_name == 'scripts' or module_name.startswith('scripts.'):
        del sys.modules[module_name]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
import torch
from scripts.common import seed_all
seed_all(42)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT = Path('/content/drive/MyDrive/AI-Crime-Investigation-FYP/canonical-four-class-v2')
except ImportError:
    OUTPUT = REPO / 'artifacts'
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Persistent output:', OUTPUT)


In [ ]:
import kagglehub
try:
    KAGGLE_DOWNLOAD = Path(kagglehub.dataset_download('mateohervas/dcsass-dataset'))
except Exception as exc:
    raise RuntimeError('Kaggle download failed. Check network/access; if authentication is required, use Colab secrets or kagglehub.login(), never commit credentials.') from exc
from scripts.dataset_paths import resolve_dataset_root
DATA = resolve_dataset_root(KAGGLE_DOWNLOAD)
print('KaggleHub returned:', KAGGLE_DOWNLOAD)
print('Verified dataset package root:', DATA)


In [ ]:
from pathlib import Path
import json
from IPython.display import display, Markdown, Image
from scripts.smoke import prepare_smoke

# Repreparation invalidates any previous in-memory approval or result.
SOURCE_MAPPING_REVIEWED = False
RUN_PROTOTYPE_TRAINING = False
SMOKE_RESULT = None
CONFIG = json.loads(Path('configs/prototype.json').read_text())
CONFIG.update(sources_per_class=3, epochs=2, batch_size=4)
PREPARATION = OUTPUT / 'smoke_preparation'
AUTOMATIC_REPORT = prepare_smoke(DATA, OUTPUT / 'inspection', PREPARATION, CONFIG)
display(Markdown((PREPARATION / 'validation.md').read_text()))
if AUTOMATIC_REPORT['automatic_status'] == 'PASS':
    display(Image(filename=str(PREPARATION / 'review_sheet.jpg')))
    print('Review all mapped sources:', PREPARATION / 'source_review.csv')
else:
    print('STOP: automatic validation failed. Fix the reported error; do not enable training.')
print('Full automatic report:', PREPARATION / 'validation.json')
print('CUDA currently available:', AUTOMATIC_REPORT['gpu_available'], AUTOMATIC_REPORT['gpu_name'])


In [ ]:
# This extends the existing preflight; it never approves source mapping or trains.
from IPython.display import HTML
from scripts.review import review_selected
TEMPORAL_REVIEW = None
TEMPORAL_REVIEW_PATH = PREPARATION / 'temporal_review' / 'review.json'
if AUTOMATIC_REPORT['automatic_status'] == 'PASS':
    TEMPORAL_REVIEW = review_selected(
        AUTOMATIC_REPORT['split_manifest'], AUTOMATIC_REPORT['data_root'],
        PREPARATION / 'temporal_review', samples=12,
    )
    display(Markdown((PREPARATION / 'temporal_review' / 'review_summary.md').read_text()))
    display(HTML((PREPARATION / 'temporal_review' / 'review.html').read_text()))
    print('Self-contained visual report (download/open in browser):', PREPARATION / 'temporal_review/review.html')
    print('Technical:', TEMPORAL_REVIEW['technical_status'], '| Human approval: PENDING')
    print('Suspicious pairs:', len(TEMPORAL_REVIEW['pairs']))
    print('Export bundle:', PREPARATION / 'temporal_review/review_artifacts.zip')
    print('This report was generated from AUTOMATIC_REPORT selected clips in this runtime, not from the synthetic demonstration.')
else:
    print('Temporal review skipped: resolve the existing preflight failures first.')


In [ ]:
# Export only review artifacts from the current report. This is NOT training approval.
DOWNLOAD_REAL_REVIEW = False  # Set only this export flag True to download the ZIP.
REAL_REVIEW_BUNDLE = PREPARATION / 'temporal_review' / 'review_artifacts.zip'
if DOWNLOAD_REAL_REVIEW:
    if TEMPORAL_REVIEW is None or not REAL_REVIEW_BUNDLE.is_file():
        raise RuntimeError('Generate the real temporal review in the preceding cell first')
    from google.colab import files
    files.download(str(REAL_REVIEW_BUNDLE))
else:
    print('Drive export location:', REAL_REVIEW_BUNDLE)
    print('To download: set DOWNLOAD_REAL_REVIEW=True and rerun ONLY this cell.')
print('Keep SOURCE_MAPPING_REVIEWED=False and RUN_PROTOTYPE_TRAINING=False during review/export.')


## Required review, then one launch cell
Review `validation.md` and `source_review.csv`, especially the selected sources and Fighting. Open original clips if previews are inconclusive: three sampled frames cannot prove a crime-category label. Confirm that original-source names match the dataset convention and that differently named videos of the same incident do not cross splits. This report automatically checks paths, canonical IDs, four-class counts, available readable sources, the recorded duplicate audit, exact source overlap and selected-frame decoding. It does not certify the semantic mapping.

The report checks saved duplicate-hash records against manifest/counts/current paths and file sizes; it does not rehash every excluded video. If dataset contents changed, rerun notebook 01. If unknown source naming, missing files, stale inspection counts, or any other FAIL appears, resolve it before proceeding. The report records current readable Fighting sources; even eight total sources yield only a tiny independent holdout.

After your review, set **both flags below to True** and run the launch cell. A PASS alone never sets either flag. If the GPU is unavailable, reconnect a GPU runtime before retrying. If you cannot verify the source mapping, leave the review flag False. Full training remains outside this launcher.

Also review temporal_review/review.html and review_cases.csv. Orange flags are uncertain shared-scene candidates, not verified incident duplicates; red exact-byte duplicates or decoding failures block the launch. Inspect the original videos and resolve suspicious cross-split cases before approval. Short/static clips and no similarity flags do not establish independence. No code sets human approval automatically.


In [ ]:
SOURCE_MAPPING_REVIEWED = False  # True ONLY after your review of this report and source mapping.
RUN_PROTOTYPE_TRAINING = False  # True ONLY for this reviewed two-epoch smoke.
SMOKE_RESULT = None  # Prevent displaying an earlier result if this invocation fails.
from scripts.smoke import run_reviewed_smoke
SMOKE_RESULT = run_reviewed_smoke(
    PREPARATION / 'validation.json', OUTPUT / 'classifier' / 'runs', CONFIG,
    source_mapping_reviewed=SOURCE_MAPPING_REVIEWED,
    run_prototype_training=RUN_PROTOTYPE_TRAINING,
    require_gpu=True,
    temporal_review_path=TEMPORAL_REVIEW_PATH,
)
if SMOKE_RESULT is None:
    print('Training disabled. Review the automatic report and mapping before enabling both flags.')


In [ ]:
# Only the result returned by THIS successful training/evaluation invocation is shown.
if SMOKE_RESULT is not None:
    print('Actual training device:', SMOKE_RESULT['actual_training_device'], SMOKE_RESULT['gpu_name'])
    print('Class order:', SMOKE_RESULT['class_order'])
    print(json.dumps(SMOKE_RESULT['metrics'], indent=2))
    print('Best checkpoint:', SMOKE_RESULT['checkpoint'])
    print('All artifacts:', SMOKE_RESULT['run_dir'])
    print(SMOKE_RESULT['limitation'])
    DOWNLOAD_CHECKPOINT = False
    if DOWNLOAD_CHECKPOINT:
        from google.colab import files
        files.download(SMOKE_RESULT['checkpoint'])
else:
    print('No new training/evaluation executed successfully in this session; no new metrics to report.')
